## Genrate the blog using LLM in langchain & lang grapgh

---
`using prompt chaining`
---
- Here we using the mutiple LLM

In [1]:
from langgraph.graph import StateGraph,START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict

In [2]:
load_dotenv()  # Load environment variables from .env file
model = ChatGroq(model="openai/gpt-oss-120b")

In [12]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation: int

In [13]:
def create_outline(state: BlogState) -> BlogState:
    
    # fetch the title from the state
    title = state['title']
    
    # call LLM to generate the outline for the blog
    prompt = f"Create a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content
    
    # update the state with the generated outline
    state['outline'] = outline
    return state

In [14]:
def create_blog(state: BlogState) -> BlogState:
    
    title = state['title']
    outline = state['outline']
    
    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'
    
    content = model.invoke(prompt).content
    
    state['content'] = content
    return state

In [ ]:
def evaluate_blog(state:BlogState) -> BlogState:
    content = state['content']
    
    prompt = f"Evaluate the following blog content for clarity, coherence, and engagement: \n {content}"
    
    evaluation = model.invoke(prompt)
    
    state['evaluation'] = evaluation
    return state

In [22]:
graph = StateGraph(BlogState)

# creating nodes for the graph
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog',evaluate_blog)

# creating edges for the graph
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [23]:
# initial state with the title of the blog
initial_state = {'title': 'The Future of AI in Healthcare'}
final_state = workflow.invoke(initial_state)
print(final_state)  
# This will print the final state with the generated blog content

{'title': 'The Future of AI in Healthcare', 'outline': '**Blog Title:** *The Future of AI in Healthcare: Transforming Medicine, Empowering Patients, and Shaping Tomorrow’s Health Systems*\n\n---\n\n## 1. Introduction  \n   - **Hook:** A striking statistic or anecdote (e.g., “In 2023, AI‑driven diagnostics reduced cancer detection time by 30%”)  \n   - **Why It Matters:** Briefly explain the stakes—rising costs, aging populations, and the need for faster, more accurate care.  \n   - **Purpose of the Post:** Outline what readers will learn (current landscape, emerging innovations, challenges, and realistic forecasts).  \n   - **Thesis Statement:** AI isn’t a distant sci‑fi fantasy; it’s already reshaping healthcare and will become an indispensable partner in clinical decision‑making, patient engagement, and system efficiency.\n\n---\n\n## 2. The Current State of AI in Healthcare  \n   ### 2.1. Core Applications Already in Use  \n   - **Medical Imaging & Radiology** – Deep learning for X‑

In [18]:
print(f"Generated outline: \n {final_state['outline']}")  # This will print the final state with the generated blog content

Generated outline: 
 **Blog Title:**  
*The Future of AI in Healthcare: Transforming Care, Research, and the Patient Experience*

---

## 1. Introduction  
- **Hook:** A striking statistic or anecdote (e.g., “In 2023, AI‑driven diagnostics cut cancer‑stage‑identification time by 40%.”)  
- **Why it matters:** Briefly explain how AI is already reshaping health‑care and why the next decade will be a tipping point.  
- **Thesis statement:** Outline the blog’s purpose – to explore emerging AI technologies, real‑world applications, challenges, and the roadmap that will define health‑care in the 2030s.

---

## 2. The Current Landscape of AI in Healthcare  
### 2.1. Core Technologies in Use Today  
- Machine learning (ML) for imaging analysis  
- Natural language processing (NLP) for clinical documentation  
- Predictive analytics for risk stratification  
- Robotics in surgery & logistics  

### 2.2. Key Success Stories  
- IBM Watson Oncology (clinical decision support)  
- Google DeepMind

In [19]:
print(f"Generated content: \n {final_state['content']}")  # This will print the final state with the generated blog content

Generated content: 
 **The Future of AI in Healthcare: Transforming Care, Research, and the Patient Experience**  

*By [Your Name], [Date]*  

---  

## 1. Introduction  

**Hook:** In 2023, AI‑driven diagnostic platforms reduced the time to identify the stage of breast cancer by **40 %**, shaving weeks off the treatment‑planning cycle and saving countless lives.  

**Why it matters:** That single statistic is a glimpse of a broader transformation already underway. From algorithms that read radiology scans faster than a human radiologist to chatbots that triage urgent‑care calls, artificial intelligence (AI) is reshaping how we prevent, diagnose, and treat disease. Yet we are still at the foot of a steep curve. The next decade promises a **tipping point** where AI moves from a helpful adjunct to an integral, “digital organ” of the health‑care system.  

**Thesis statement:** This blog explores the emerging AI technologies that will define health‑care in the 2030s, showcases real‑world

##### `Note:` When we use only langchain, we didn't access the outline of the blog, but with langgraph we can access the outline of the blog as well.

In [24]:
print(f"Evaluation:  \n {final_state['evaluation']}")  
# This will print the final state with the generated blog content

Evaluation:  
 ## Quick‑Read Verdict  

| Aspect | Score (1‑5) | Why it Scores That Way |
|--------|-------------|------------------------|
| **Clarity** | **4.5** | The writing is jargon‑light, concepts are defined, and tables/boxes break up dense information. A few sentences are long‑winded or could use simpler phrasing. |
| **Coherence** | **4.2** | The piece follows a logical “problem → present → future → benefits → barriers → case studies → roadmap” arc, but the transition from **Section 5** (Challenges) to **Section 6** (Case Studies) could be smoother. |
| **Engagement** | **4.0** | Strong hook, real‑world numbers, and vivid case‑study snippets keep the reader interested. Adding more storytelling (e.g., a clinician’s “day‑in‑the‑life” vignette) would raise the emotional pull. |

Overall, the blog is **well‑structured, data‑rich, and persuasive**, but a few tweaks can make it even more readable and compelling for a mixed audience of clinicians, administrators, and tech‑savvy read